In [8]:
import os
import json
import time
import requests
import subprocess
from typing import Dict, List, Any, Optional, Tuple, TypedDict

from langchain.chat_models import ChatOpenAI
from langchain.tools import Tool
from langchain.schema import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [16]:
class ServiceManager:
    def __init__(self, config_path = "api_services.json"):
        with open (config_path, "r") as f:
            self.config = json.load(f)
        self.services = {}
        self.running_processes = {}
    
    def ensure_services_running(self, service_name: str) -> str:
        "Ensure a service is running and return its base URL"
        if service_name not in self.config:
            raise ValueError(f"Unknown service: {service_name}")
        
        service_config = self.config[service_name]
        port = service_config["port"]
        base_url = f"http://localhost:{port}"
        
        try:
            response = requests.get(f"{base_url}/health")
            if response.status_code == 200:
                return base_url
        except requests.RequestException:
            pass
        
        # If the service is not running, we start it
        if service_name not in self.running_processes:
            app_module = service_config["app_module"]
            print(f"Starting {service_name} service on port {port}...")
            
            process = subprocess.Popen(
                ["uvicorn", f"{app_module}:app", "--port", str(port)],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE
            )
            
            self.running_processes[service_name] = process
            
            # Wait for the service to start
            for _ in range(10):
                time.sleep(2)
                try:
                    response = requests.get(f"{base_url}/health")
                    if response.status_code == 200:
                        print(f"{service_name} service is running now.")
                        return base_url
                except requests.RequestException:
                    continue
                
            print(f"Failed to start {service_name} service.")
            return None
        
        return base_url
        
    def stop_all_services(self):
        for name, process in self.running_processes.items():
            print(f"Stopping {name} services..")
            process.terminate()
            process.wait(timeout = 5)
        self.running_processes = {}

In [ ]:
# import os
# import json
# import time
# import requests
# import subprocess
# from typing import Dict, List, Any, Optional, Tuple, TypedDict

# from langchain.chat_models import ChatOpenAI
# from langchain.tools import Tool
# from langchain.schema import SystemMessage, HumanMessage, AIMessage
# from langgraph.graph import StateGraph, START, END
# from langgraph.graph.message import add_messages

# # Service Manager for API Services
# class ServiceManager:
#     def __init__(self, config_path="api_services.json"):
#         with open(config_path, "r") as f:
#             self.config = json.load(f)
#         self.services = {}
#         self.running_processes = {}
        
#     def ensure_service_running(self, service_name: str) -> str:
#         """Ensure a service is running and return its base URL"""
#         if service_name not in self.config:
#             raise ValueError(f"Unknown service: {service_name}")
            
#         service_config = self.config[service_name]
#         port = service_config["port"]
#         base_url = f"http://localhost:{port}"
        
#         # Check if service is already running
#         try:
#             response = requests.get(f"{base_url}/health")
#             if response.status_code == 200:
#                 return base_url
#         except requests.RequestException:
#             pass
            
#         # Service not running, start it
#         if service_name not in self.running_processes:
#             app_module = service_config["app_module"]
#             print(f"Starting {service_name} service on port {port}...")
            
#             process = subprocess.Popen(
#                 ["uvicorn", f"{app_module}:app", "--port", str(port)],
#                 stdout=subprocess.PIPE,
#                 stderr=subprocess.PIPE
#             )
            
#             self.running_processes[service_name] = process
            
#             # Wait for service to start
#             for _ in range(10):  # Try for about 20 seconds
#                 time.sleep(2)
#                 try:
#                     response = requests.get(f"{base_url}/health")
#                     if response.status_code == 200:
#                         print(f"{service_name} service is now running.")
#                         return base_url
#                 except requests.RequestException:
#                     continue
                    
#             print(f"Failed to start {service_name} service.")
#             return None
        
#         return base_url
    
#     def stop_all_services(self):
#         """Stop all running services"""
#         for name, process in self.running_processes.items():
#             print(f"Stopping {name} service...")
#             process.terminate()
#             process.wait(timeout=5)
#         self.running_processes = {}

# # State definition for LangGraph
# class AgentState(TypedDict):
#     messages: List[Dict[str, Any]]
#     weather_data: Optional[Dict[str, Any]]
#     news_data: Optional[Dict[str, Any]]
#     job_data: Optional[Dict[str, Any]]

# # Initialize service manager
# service_manager = ServiceManager()

# # Define API tools
# def get_weather(location: str = "Seattle") -> Dict[str, Any]:
#     """Get weather forecast for a location"""
#     base_url = service_manager.ensure_service_running("weather")
#     if not base_url:
#         return {"error": "Weather service unavailable"}
    
#     try:
#         response = requests.get(f"{base_url}/weather/47.69,-122.1808,7")
#         if response.status_code == 200:
#             return response.json()
#         else:
#             return {"error": f"Weather API error: {response.status_code}"}
#     except requests.RequestException as e:
#         return {"error": f"Request failed: {str(e)}"}

# def get_news(query: str, category: str = None) -> Dict[str, Any]:
#     """Get news articles based on a query"""
#     base_url = service_manager.ensure_service_running("news")
#     if not base_url:
#         return {"error": "News service unavailable"}
    
#     params = {"keywords": query}
#     if category:
#         params["category"] = category
    
#     try:
#         response = requests.get(f"{base_url}/news", params=params)
#         if response.status_code == 200:
#             return response.json()
#         else:
#             return {"error": f"News API error: {response.status_code}"}
#     except requests.RequestException as e:
#         return {"error": f"Request failed: {str(e)}"}

# def search_jobs(job_title: str, location: str = None) -> Dict[str, Any]:
#     """Search for jobs based on title and location"""
#     base_url = service_manager.ensure_service_running("jobs")
#     if not base_url:
#         return {"error": "Jobs service unavailable"}
    
#     params = {"job_title": job_title}
#     if location:
#         params["location"] = location
    
#     try:
#         response = requests.get(f"{base_url}/jobs", params=params)
#         if response.status_code == 200:
#             return response.json()
#         else:
#             return {"error": f"Jobs API error: {response.status_code}"}
#     except requests.RequestException as e:
#         return {"error": f"Request failed: {str(e)}"}

# # Create tools
# weather_tool = Tool.from_function(
#     func=get_weather,
#     name="get_weather",
#     description="Get weather forecast for a location"
# )

# news_tool = Tool.from_function(
#     func=get_news,
#     name="get_news",
#     description="Get news articles based on a query"
# )

# jobs_tool = Tool.from_function(
#     func=search_jobs,
#     name="search_jobs",
#     description="Search for jobs based on title and location"
# )

# # Initialize LLM
# llm = ChatOpenAI(temperature=0)

# # LangGraph Nodes
# def agent(state: AgentState) -> Dict:
#     """Main agent node that processes messages and decides on actions"""
#     messages = state["messages"]
    
#     # Prepare messages for LLM
#     llm_messages = [
#         SystemMessage(content="""You are a helpful assistant with access to weather, news, and job search tools.
#         If the user asks about weather, use the get_weather tool.
#         If the user asks about news, use the get_news tool.
#         If the user asks about jobs, use the search_jobs tool.
#         Always try to provide helpful and concise responses."""),
#     ]
    
#     # Add conversation history
#     for message in messages:
#         if message["role"] == "user":
#             llm_messages.append(HumanMessage(content=message["content"]))
#         else:
#             llm_messages.append(AIMessage(content=message["content"]))
    
#     # Get response from LLM
#     tools = [weather_tool, news_tool, jobs_tool]
#     response = llm.predict_messages(llm_messages, tools=tools)
    
#     # Check if tool use is needed
#     if response.tool_calls:
#         tool_call = response.tool_calls[0]
#         tool_name = tool_call.name
#         tool_args = json.loads(tool_call.args)
        
#         if tool_name == "get_weather":
#             return {"next": "weather_node"}
#         elif tool_name == "get_news":
#             state["tool_args"] = tool_args
#             return {"next": "news_node"}
#         elif tool_name == "search_jobs":
#             state["tool_args"] = tool_args
#             return {"next": "jobs_node"}
    
#     # No tool use, just respond directly
#     state["messages"].append({"role": "assistant", "content": response.content})
#     return {"next": END}

# def weather_node(state: AgentState) -> Dict:
#     """Node for handling weather requests"""
#     weather_data = get_weather()
#     state["weather_data"] = weather_data
    
#     # Format a nice response
#     if "error" in weather_data:
#         response = f"Sorry, I couldn't get the weather information: {weather_data['error']}"
#     else:
#         location = weather_data.get("location", "the requested location")
#         forecast = weather_data.get("forecast", [])
#         if forecast:
#             today = forecast[0]
#             response = f"Weather for {location}: {today.get('description', 'No data available')}"
#         else:
#             response = f"Weather data for {location} is available, but no forecast details were found."
    
#     state["messages"].append({"role": "assistant", "content": response})
#     return {"next": END}

# def news_node(state: AgentState) -> Dict:
#     """Node for handling news requests"""
#     tool_args = state.get("tool_args", {})
#     query = tool_args.get("query", "technology")
#     category = tool_args.get("category")
    
#     news_data = get_news(query, category)
#     state["news_data"] = news_data
    
#     # Format a nice response
#     if "error" in news_data:
#         response = f"Sorry, I couldn't get the news: {news_data['error']}"
#     else:
#         articles = news_data.get("result", [])
#         if articles:
#             response = f"Here are some recent news articles about {query}:\n\n"
#             for i, article in enumerate(articles[:3], 1):
#                 response += f"{i}. {article.get('title')}\n"
#                 if article.get('description'):
#                     response += f"   {article.get('description')[:100]}...\n"
#             response += f"\nFound {len(articles)} articles in total."
#         else:
#             response = f"I didn't find any news articles about {query}."
    
#     state["messages"].append({"role": "assistant", "content": response})
#     return {"next": END}

# def jobs_node(state: AgentState) -> Dict:
#     """Node for handling job search requests"""
#     tool_args = state.get("tool_args", {})
#     job_title = tool_args.get("job_title", "Data Scientist")
#     location = tool_args.get("location")
    
#     jobs_data = search_jobs(job_title, location)
#     state["jobs_data"] = jobs_data
    
#     # Format a nice response
#     if "error" in jobs_data:
#         response = f"Sorry, I couldn't search for jobs: {jobs_data['error']}"
#     else:
#         jobs = jobs_data.get("result", [])
#         if jobs:
#             location_str = f" in {location}" if location else ""
#             response = f"Here are some recent {job_title} jobs{location_str}:\n\n"
#             for i, job in enumerate(jobs[:3], 1):
#                 response += f"{i}. {job.get('title')} at {job.get('company')}\n"
#                 response += f"   Location: {job.get('location')}\n"
#                 if job.get('salary'):
#                     response += f"   Salary: {job.get('salary')}\n"
#             response += f"\nFound {len(jobs)} jobs in total."
#         else:
#             response = f"I didn't find any {job_title} jobs{' in ' + location if location else ''}."
    
#     state["messages"].append({"role": "assistant", "content": response})
#     return {"next": END}

# # Create LangGraph
# workflow = StateGraph(AgentState)

# # Add nodes
# workflow.add_node("agent", agent)
# workflow.add_node("weather_node", weather_node)
# workflow.add_node("news_node", news_node)
# workflow.add_node("jobs_node", jobs_node)

# # Add edges
# workflow.add_edge(START, "agent")
# workflow.add_edge("agent", END)
# workflow.add_edge("agent", "weather_node")
# workflow.add_edge("agent", "news_node")
# workflow.add_edge("agent", "jobs_node")
# workflow.add_edge("weather_node", END)
# workflow.add_edge("news_node", END)
# workflow.add_edge("jobs_node", END)

# # Compile the graph
# app = workflow.compile()

# # Helper function to run a conversation
# def chat(user_input: str, state: Optional[Dict] = None) -> Dict:
#     """Process a user message and return the updated state"""
#     if state is None:
#         state = {"messages": []}
    
#     # Add user message
#     state["messages"].append({"role": "user", "content": user_input})
    
#     # Process through graph
#     result = app.invoke(state)
    
#     return result

# # Example usage
# if __name__ == "__main__":
#     try:
#         # Create a config file for API services
#         config = {
#             "weather": {
#                 "app_module": "weather_app",
#                 "port": 8001
#             },
#             "news": {
#                 "app_module": "news_app",
#                 "port": 8002
#             },
#             "jobs": {
#                 "app_module": "jobs_app", 
#                 "port": 8003
#             }
#         }
        
#         with open("api_services.json", "w") as f:
#             json.dump(config, f, indent=2)
        
#         # Start conversation
#         state = {"messages": []}
        
#         print("AI Assistant: Hello! I can help you with weather, news, and job searches. What would you like to know?")
        
#         while True:
#             user_input = input("You: ")
#             if user_input.lower() in ["exit", "quit", "bye"]:
#                 break
                
#             state = chat(user_input, state)
            
#             # Print assistant's response
#             for message in state["messages"]:
#                 if message["role"] == "assistant":
#                     print(f"AI Assistant: {message['content']}")
    
#     finally:
#         # Clean up by stopping all services
#         service_manager.stop_all_services()